# Prepare Augmenterra data for comparison

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import re
import pyarrow as pa
import pyarrow.parquet as pq
import gc
import os
import psutil
import pyogrio
import shutil

## Configuration

In [19]:
# --------------------------------------------------
# INPUT
# --------------------------------------------------

# ASC or DSC
orbit = "ASC"

# Path to the Augmenterra folder
input_dir = Path(
    f"/home/lukasscharf/projects/ADUCAT/data/Augmenterra/{orbit}"
)

print(f"Input directory:  {input_dir}")

shapefiles = sorted(input_dir.glob("*.shp"))

print(f"Found {len(shapefiles)} shapefiles:")

for shp in shapefiles:
    print(shp.name)

# Output directory
output_dir = Path(
    "/home/lukasscharf/projects/ADUCAT/data/Augmenterra/processed")

output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output dir:{output_dir}")



Input directory:  /home/lukasscharf/projects/ADUCAT/data/Augmenterra/ASC
Found 9 shapefiles:
AT_VIENNA_TUNNELLING_SNT_T73_A_1_ES13437A001S.shp
AT_VIENNA_TUNNELLING_SNT_T73_A_2_ES13437A001S.shp
AT_VIENNA_TUNNELLING_SNT_T73_A_3_ES13437A001S.shp
AT_VIENNA_TUNNELLING_SNT_T73_A_4_ES13437A001S.shp
AT_VIENNA_TUNNELLING_SNT_T73_A_5_ES13437A001S.shp
AT_VIENNA_TUNNELLING_SNT_T73_A_6_ES13437A001S.shp
AT_VIENNA_TUNNELLING_SNT_T73_A_7_ES13437A001S.shp
AT_VIENNA_TUNNELLING_SNT_T73_A_8_ES13437A001S.shp
AT_VIENNA_TUNNELLING_SNT_T73_A_9_ES13437A001S.shp
Output dir:/home/lukasscharf/projects/ADUCAT/data/Augmenterra/processed


## Functions

In [20]:
def add_velocity(gdf, intervals):
    """
    Calculate velocity and velocity standard deviation
    for the supplied GeoDataFrame chunk.

    The GeoDataFrame passed to this function should already
    be a limited chunk of the input data.

    The regression is identical to the original implementation:
        np.polyfit(t, y, 1, cov=True)
    """

    # --------------------------------------------------
    # Find time-series columns
    # --------------------------------------------------

    ts_cols = [
        c for c in gdf.columns
        if re.fullmatch(r"D\d{8}", c)
    ]

    dates = pd.to_datetime(
        [c[1:] for c in ts_cols],
        format="%Y%m%d"
    )

    # --------------------------------------------------
    # Process each interval
    # --------------------------------------------------

    for start_date, end_date in intervals:

        print(
            f"    Processing "
            f"{start_date} → {end_date}"
        )

        start_date = pd.Timestamp(start_date)
        end_date = pd.Timestamp(end_date)

        # Select observations in this interval
        mask = (
            (dates >= start_date) &
            (dates <= end_date)
        )

        selected_cols = [
            c
            for c, keep in zip(ts_cols, mask)
            if keep
        ]

        selected_dates = dates[mask]

        if len(selected_dates) < 3:

            print(
                "      Skipping: "
                "fewer than 3 observations"
            )

            continue

        # --------------------------------------------------
        # Time in years
        # --------------------------------------------------

        t = (
            selected_dates - selected_dates[0]
        ).total_seconds() / (
            365.25 * 24 * 3600
        )

        t = np.asarray(
            t,
            dtype=np.float64
        )

        # --------------------------------------------------
        # Output arrays
        # --------------------------------------------------

        velocities = np.full(
            len(gdf),
            np.nan,
            dtype=np.float64
        )

        velocity_std = np.full(
            len(gdf),
            np.nan,
            dtype=np.float64
        )

        # --------------------------------------------------
        # Process every point in this chunk
        # --------------------------------------------------

        for local_i, (_, row) in enumerate(
            gdf.iterrows()
        ):

            y = pd.to_numeric(
                row[selected_cols],
                errors="coerce"
            ).to_numpy(
                dtype=np.float64
            )

            valid = np.isfinite(y)

            if valid.sum() >= 3:

                # Perform ordinary least-squares linear regression:
                # displacement = velocity * time + offset
                # coefficients[0] is the fitted velocity (slope) [mm/year].
                # covariance[0,0] is the estimated variance of this velocity.
                coefficients, covariance = np.polyfit(
                    t[valid],
                    y[valid],
                    1,
                    cov=True
                )

                # Estimated mean linear velocity [mm/year]
                velocities[local_i] = round(
                    coefficients[0],
                    1
                )

                # Standard error of the estimated linear velocity [mm/year].
                # It is derived from the covariance matrix of the regression:
                # velocity standard error = sqrt(variance of fitted slope)
                # This is not necessarily the same formular as Augmenterra is using
                velocity_std[local_i] = round(
                    np.sqrt(covariance[0, 0]),
                    1
                )

            del y
            del valid

        # --------------------------------------------------
        # Add result columns
        # --------------------------------------------------

        start_str = start_date.strftime("%Y%m%d")
        end_str = end_date.strftime("%Y%m%d")

        vel_col = (
            f"VEL_{start_str}_{end_str}"
        )

        std_col = (
            f"V_STDERR_{start_str}_{end_str}"
        )

        gdf[vel_col] = velocities
        gdf[std_col] = velocity_std

        print(
            f"      Added: {vel_col}"
        )

        print(
            f"      Added: {std_col}"
        )

        # --------------------------------------------------
        # Clean interval-specific memory
        # --------------------------------------------------

        del velocities
        del velocity_std
        del selected_cols
        del selected_dates
        del mask
        del t

        gc.collect()

    # --------------------------------------------------
    # Clean function-level objects
    # --------------------------------------------------

    del ts_cols
    del dates

    gc.collect()

    return gdf

In [21]:
process = psutil.Process(os.getpid())


def print_memory(label=""):

    rss_gb = (
        process.memory_info().rss
        / (1024 ** 3)
    )

    print(
        f"[MEMORY] {label}: "
        f"{rss_gb:.2f} GB RSS"
    )

## Recalculate velocities

In [22]:
# Define for which intervals you want to recalculate the mean velocities

intervals = [
    ("2022-01-01", "2024-12-31"),
    ("2020-01-01", "2024-12-31"),
    ("2019-01-01", "2026-04-30")
]

## Run through all shapefiles

In [23]:
# ============================================================
# SETTINGS
# ============================================================

# FOR TESTING: create a subset of the shapefile list
#shapefiles=shapefiles[:2]

CHUNK_SIZE = 20000

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Temporary chunks
temp_dir = output_dir / "temp_chunks"

temp_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Final output
final_output = (
    output_dir /
    f"VIENNA_TUNNELLING_velocity_{orbit}.parquet"
)

print("Final file name:", final_output)


Final file name: /home/lukasscharf/projects/ADUCAT/data/Augmenterra/processed/VIENNA_TUNNELLING_velocity_ASC.parquet


### Process the shapefiles and create one parquet file with the new calculated mean velocities

In [24]:
# ============================================================
# PROCESS SHAPEFILES
# ============================================================

temp_files = []

global_chunk_number = 0


for shp_index, shp in enumerate(
    shapefiles,
    start=1
):

    print()
    print("=" * 70)
    print(
        f"SHAPEFILE {shp_index}/{len(shapefiles)}"
    )
    print(shp.name)
    print("=" * 70)

    # --------------------------------------------------------
    # Determine number of features
    # --------------------------------------------------------

    info = pyogrio.read_info(shp)

    feature_count = info["features"]

    print(
        f"Total points: "
        f"{feature_count:,}"
    )

    print(
        f"Chunk size: "
        f"{CHUNK_SIZE:,}"
    )

    number_of_chunks = (
        feature_count + CHUNK_SIZE - 1
    ) // CHUNK_SIZE

    print(
        f"Chunks: "
        f"{number_of_chunks:,}"
    )

    # --------------------------------------------------------
    # Process chunks
    # --------------------------------------------------------

    for chunk_index, start in enumerate(
        range(
            0,
            feature_count,
            CHUNK_SIZE
        ),
        start=1
    ):

        end = min(
            start + CHUNK_SIZE,
            feature_count
        )

        print()
        print(
            f"Chunk "
            f"{chunk_index}/{number_of_chunks}"
            f" | points "
            f"{start:,}–{end - 1:,}"
        )

        print_memory(
            "before reading chunk"
        )

        # ----------------------------------------------------
        # READ ONLY THIS CHUNK
        # ----------------------------------------------------

        gdf_chunk = pyogrio.read_dataframe(
            shp,
            skip_features=start,
            max_features=CHUNK_SIZE
        )

        print(
            f"Loaded: "
            f"{len(gdf_chunk):,} points"
        )

        print_memory(
            "after reading chunk"
        )

        # ----------------------------------------------------
        # CALCULATE VELOCITIES
        # ----------------------------------------------------

        gdf_chunk = add_velocity(
            gdf_chunk,
            intervals
        )

        print_memory(
            "after velocity calculation"
        )

        # ----------------------------------------------------
        # WRITE TEMPORARY PARQUET
        # ----------------------------------------------------

        global_chunk_number += 1

        temp_file = (
            temp_dir /
            f"part_{global_chunk_number:05d}.parquet"
        )

        gdf_chunk.to_parquet(
            temp_file,
            index=False,
            compression="zstd"
        )

        temp_files.append(temp_file)

        print(
            f"Written: "
            f"{temp_file.name}"
        )

        print_memory(
            "after writing chunk"
        )

        # ----------------------------------------------------
        # RELEASE CHUNK
        # ----------------------------------------------------

        del gdf_chunk

        gc.collect()

        print_memory(
            "after chunk cleanup"
        )


SHAPEFILE 1/9
AT_VIENNA_TUNNELLING_SNT_T73_A_1_ES13437A001S.shp
Total points: 19,922
Chunk size: 20,000
Chunks: 1

Chunk 1/1 | points 0–19,921
[MEMORY] before reading chunk: 0.75 GB RSS
Loaded: 19,922 points
[MEMORY] after reading chunk: 0.83 GB RSS
    Processing 2022-01-01 → 2024-12-31
      Added: VEL_20220101_20241231
      Added: V_STDERR_20220101_20241231
    Processing 2020-01-01 → 2024-12-31
      Added: VEL_20200101_20241231
      Added: V_STDERR_20200101_20241231
    Processing 2019-01-01 → 2026-04-30
      Added: VEL_20190101_20260430
      Added: V_STDERR_20190101_20260430
[MEMORY] after velocity calculation: 0.88 GB RSS
Written: part_00001.parquet
[MEMORY] after writing chunk: 0.88 GB RSS
[MEMORY] after chunk cleanup: 0.78 GB RSS

SHAPEFILE 2/9
AT_VIENNA_TUNNELLING_SNT_T73_A_2_ES13437A001S.shp
Total points: 13,176
Chunk size: 20,000
Chunks: 1

Chunk 1/1 | points 0–13,175
[MEMORY] before reading chunk: 0.78 GB RSS
Loaded: 13,176 points
[MEMORY] after reading chunk: 0.82 GB

In [25]:
# ============================================================
# COMBINE ALL TEMPORARY PARQUETS
# ============================================================

print()
print("=" * 70)
print("COMBINING TEMPORARY PARQUETS")
print("=" * 70)

print(
    f"Temporary files: "
    f"{len(temp_files)}"
)

print(
    f"Final output: "
    f"{final_output}"
)


# ------------------------------------------------------------
# Use first file to establish schema
# ------------------------------------------------------------

first_file = temp_files[0]

reader = pq.ParquetFile(
    first_file
)

schema = reader.schema_arrow

writer = pq.ParquetWriter(
    final_output,
    schema,
    compression="zstd"
)


# ------------------------------------------------------------
# Append every temporary Parquet
# ------------------------------------------------------------

for i, temp_file in enumerate(
    temp_files,
    start=1
):

    print(
        f"Combining "
        f"{i}/{len(temp_files)}: "
        f"{temp_file.name}"
    )

    parquet_file = pq.ParquetFile(
        temp_file
    )

    table = parquet_file.read()

    writer.write_table(
        table
    )

    del table
    del parquet_file
    

    gc.collect()


# ------------------------------------------------------------
# Close writer
# ------------------------------------------------------------

writer.close()

print()
print("=" * 70)
print("FINAL PARQUET CREATED")
print("=" * 70)

print(final_output)

print_memory(
    "after final parquet"
)


COMBINING TEMPORARY PARQUETS
Temporary files: 150
Final output: /home/lukasscharf/projects/ADUCAT/data/Augmenterra/processed/VIENNA_TUNNELLING_velocity_ASC.parquet
Combining 1/150: part_00001.parquet
Combining 2/150: part_00002.parquet
Combining 3/150: part_00003.parquet
Combining 4/150: part_00004.parquet
Combining 5/150: part_00005.parquet
Combining 6/150: part_00006.parquet
Combining 7/150: part_00007.parquet
Combining 8/150: part_00008.parquet
Combining 9/150: part_00009.parquet
Combining 10/150: part_00010.parquet
Combining 11/150: part_00011.parquet
Combining 12/150: part_00012.parquet
Combining 13/150: part_00013.parquet
Combining 14/150: part_00014.parquet
Combining 15/150: part_00015.parquet
Combining 16/150: part_00016.parquet
Combining 17/150: part_00017.parquet
Combining 18/150: part_00018.parquet
Combining 19/150: part_00019.parquet
Combining 20/150: part_00020.parquet
Combining 21/150: part_00021.parquet
Combining 22/150: part_00022.parquet
Combining 23/150: part_00023.p

In [26]:
# ============================================================
# REMOVE TEMPORARY FILES
# ============================================================

if final_output.exists():

    print()
    print("Removing temporary files...")

    shutil.rmtree(temp_dir)

    print(
        f"Removed temporary directory: "
        f"{temp_dir}"
    )

else:

    print()
    print(
        "WARNING: Final output does not exist."
    )

    print(
        "Temporary files were NOT deleted."
    )


Removing temporary files...
Removed temporary directory: /home/lukasscharf/projects/ADUCAT/data/Augmenterra/processed/temp_chunks


## Seperate to velocity and timeseries file

In [27]:
velocity_file = (
    output_dir /
    f"VIENNA_TUNNELLING_velocity_{orbit}_without_timeseries.parquet"
)

timeseries_file = (
    output_dir /
    f"VIENNA_TUNNELLING_velocity_{orbit}_just_timeseries.parquet"
)

print(velocity_file)
print(timeseries_file)

/home/lukasscharf/projects/ADUCAT/data/Augmenterra/processed/VIENNA_TUNNELLING_velocity_ASC_without_timeseries.parquet
/home/lukasscharf/projects/ADUCAT/data/Augmenterra/processed/VIENNA_TUNNELLING_velocity_ASC_just_timeseries.parquet


In [28]:
# --------------------------------------------------
# full Parquet file
# --------------------------------------------------

pf = pq.ParquetFile(final_output)

columns = pf.schema_arrow.names

In [29]:
# --------------------------------------------------
# Identify time-series columns
# --------------------------------------------------

ts_cols = [
    c for c in columns
    if re.fullmatch(r"D\d{8}", c)
]

In [30]:
# CODE is kept in both files
# --------------------------------------------------

timeseries_columns = [
    "CODE",
    *ts_cols
]

In [31]:
# Everything else goes into spatial/velocity file
# --------------------------------------------------

velocity_columns = [
    c for c in columns
    if c not in ts_cols
]

In [32]:
print("Total columns:", len(columns))
print("Time-series columns:", len(ts_cols))
print("Velocity/static columns:", len(velocity_columns))

Total columns: 365
Time-series columns: 343
Velocity/static columns: 22


In [33]:
# --------------------------------------------------
# Create writers
# --------------------------------------------------

velocity_writer = None
timeseries_writer = None

In [34]:
# ============================================================
# PROCESS BATCHES
# ============================================================

for batch_number, batch in enumerate(
    pf.iter_batches(
        batch_size=CHUNK_SIZE,
        columns=velocity_columns
    ),
    start=1
):

    print(
        f"Velocity batch "
        f"{batch_number}: "
        f"{batch.num_rows:,} rows"
    )

    # RecordBatch → Table
    table = pa.Table.from_batches(
        [batch]
    )

    if velocity_writer is None:

        velocity_writer = pq.ParquetWriter(
            velocity_file,
            table.schema,
            compression="zstd"
        )

    velocity_writer.write_table(
        table
    )

    del table
    del batch

    gc.collect()


# ============================================================
# CLOSE VELOCITY WRITER
# ============================================================

if velocity_writer is not None:
    velocity_writer.close()


# ============================================================
# TIME-SERIES
# ============================================================

print()
print("Creating time-series Parquet...")


for batch_number, batch in enumerate(
    pf.iter_batches(
        batch_size=CHUNK_SIZE,
        columns=timeseries_columns
    ),
    start=1
):

    print(
        f"Time-series batch "
        f"{batch_number}: "
        f"{batch.num_rows:,} rows"
    )

    # RecordBatch → Table
    table = pa.Table.from_batches(
        [batch]
    )

    if timeseries_writer is None:

        timeseries_writer = pq.ParquetWriter(
            timeseries_file,
            table.schema,
            compression="zstd"
        )

    timeseries_writer.write_table(
        table
    )

    del table
    del batch

    gc.collect()


# ============================================================
# CLOSE TIME-SERIES WRITER
# ============================================================

if timeseries_writer is not None:
    timeseries_writer.close()


# ============================================================
# DONE
# ============================================================

print()
print("=" * 60)
print("Finished")
print("=" * 60)

print(
    f"Velocity file:\n"
    f"{velocity_file}"
)

print(
    f"Time-series file:\n"
    f"{timeseries_file}"
)

Velocity batch 1: 20,000 rows
Velocity batch 2: 20,000 rows
Velocity batch 3: 20,000 rows
Velocity batch 4: 20,000 rows
Velocity batch 5: 20,000 rows
Velocity batch 6: 20,000 rows
Velocity batch 7: 20,000 rows
Velocity batch 8: 20,000 rows
Velocity batch 9: 20,000 rows
Velocity batch 10: 20,000 rows
Velocity batch 11: 20,000 rows
Velocity batch 12: 20,000 rows
Velocity batch 13: 20,000 rows
Velocity batch 14: 20,000 rows
Velocity batch 15: 20,000 rows
Velocity batch 16: 20,000 rows
Velocity batch 17: 20,000 rows
Velocity batch 18: 20,000 rows
Velocity batch 19: 20,000 rows
Velocity batch 20: 20,000 rows
Velocity batch 21: 20,000 rows
Velocity batch 22: 20,000 rows
Velocity batch 23: 20,000 rows
Velocity batch 24: 20,000 rows
Velocity batch 25: 20,000 rows
Velocity batch 26: 20,000 rows
Velocity batch 27: 20,000 rows
Velocity batch 28: 20,000 rows
Velocity batch 29: 20,000 rows
Velocity batch 30: 20,000 rows
Velocity batch 31: 20,000 rows
Velocity batch 32: 20,000 rows
Velocity batch 33